In [10]:
# Install the library that lets pandas read .xlsx (Excel) files
%pip install openpyxl pandas numpy

Note: you may need to restart the kernel to use updated packages.


In [11]:
import pandas as pd
import numpy as np

# Read the Excel file into a table (DataFrame) called df
# This can take 30 to 90 seconds, because the file has 300,000 rows
df = pd.read_excel("dataset for decision tree.xlsx")

# Drop the id column, because it is only a row number
df = df.drop("id", axis=1)

# Show the size and the first 5 rows
print("Shape:", df.shape)
print(df.head())

Shape: (300000, 10)
  bin_3 bin_4  nom_0      nom_1    nom_2    nom_3     nom_4        ord_1  \
0     T     Y  Green   Triangle    Snake  Finland   Bassoon  Grandmaster   
1     T     Y  Green  Trapezoid  Hamster   Russia     Piano  Grandmaster   
2     F     Y   Blue  Trapezoid     Lion   Russia  Theremin       Expert   
3     F     Y    Red  Trapezoid    Snake   Canada      Oboe  Grandmaster   
4     F     N    Red  Trapezoid     Lion   Canada      Oboe  Grandmaster   

         ord_2  target  
0         Cold       0  
1          Hot       0  
2     Lava Hot       0  
3  Boiling Hot       1  
4     Freezing       0  


In [12]:
# ---------- TRAIN / TEST SPLIT (80% / 20%) ----------
# RandomState(42) makes the random shuffle the same every time
rng = np.random.RandomState(42)

# Shuffle the row numbers 0..299999 into a random order
shuffled = rng.permutation(len(df))

# The first 80% of the shuffled rows are for training, the rest for testing
cut = int(0.8 * len(df))
train_df = df.iloc[shuffled[:cut]].reset_index(drop=True)
test_df = df.iloc[shuffled[cut:]].reset_index(drop=True)

print("Train size:", len(train_df))
print("Test size:", len(test_df))


# ---------- IMPURITY FUNCTIONS ----------
def gini(labels):
    """Gini impurity of a group of labels: 1 - sum(p^2). 0 = pure."""
    # p = the proportion of each class in the group (for example 0.69 and 0.31)
    p = labels.value_counts(normalize=True).values
    return float(1 - np.sum(p ** 2))

def entropy(labels):
    """Entropy of a group of labels: -sum(p * log2(p)). 0 = pure."""
    p = labels.value_counts(normalize=True).values
    # "+ 0.0" avoids printing -0.0 for a pure group
    return float(-np.sum(p * np.log2(p)) + 0.0)


# ---------- QUICK TESTS ----------
# A group with half 0s and half 1s: the most mixed possible (2 classes)
mixed = pd.Series([0, 0, 1, 1])
# A group with only 0s: pure
pure = pd.Series([0, 0, 0, 0])

print()
print("Mixed group -> Gini:", gini(mixed), "| Entropy:", entropy(mixed))
print("Pure group  -> Gini:", gini(pure), "| Entropy:", entropy(pure))
print("Whole training target -> Gini:", round(gini(train_df["target"]), 4),
      "| Entropy:", round(entropy(train_df["target"]), 4))

Train size: 240000
Test size: 60000

Mixed group -> Gini: 0.5 | Entropy: 1.0
Pure group  -> Gini: 0.0 | Entropy: 0.0
Whole training target -> Gini: 0.4248 | Entropy: 0.8886


In [13]:
def split_gain(data, feature, impurity_fn):
    """How much does splitting `data` on `feature` reduce the impurity?
    impurity_fn is either gini or entropy."""

    # Number of rows in this node
    n = len(data)

    # Impurity BEFORE the split (the whole node)
    before = impurity_fn(data["target"])

    # Impurity AFTER the split: one group per category value of the feature
    after = 0.0
    for value, group in data.groupby(feature):
        # Weight = the fraction of the rows that went into this branch
        weight = len(group) / n
        after = after + weight * impurity_fn(group["target"])

    # Gain = how much the impurity went down
    return before - after


# The list of feature columns (every column except the target)
features = [c for c in train_df.columns if c != "target"]

# Test: the gain of each feature at the ROOT (using all training data),
# once with Gini and once with Entropy
for name, fn in [("GINI", gini), ("ENTROPY (Information Gain)", entropy)]:
    print(name)
    gains = {f: split_gain(train_df, f, fn) for f in features}
    # Print from the best feature to the worst
    for f, g in sorted(gains.items(), key=lambda item: -item[1]):
        print("   ", f, ":", round(g, 5))
    print()

GINI
    ord_2 : 0.01002
    ord_1 : 0.00907
    nom_0 : 0.00414
    nom_4 : 0.00329
    nom_1 : 0.00215
    nom_3 : 0.00212
    nom_2 : 0.00186
    bin_4 : 0.00086
    bin_3 : 2e-05

ENTROPY (Information Gain)
    ord_2 : 0.01709
    ord_1 : 0.01525
    nom_0 : 0.00719
    nom_4 : 0.0057
    nom_1 : 0.00368
    nom_3 : 0.00365
    nom_2 : 0.00319
    bin_4 : 0.00145
    bin_3 : 3e-05



In [14]:
def build_tree(data, features_left, impurity_fn, depth, max_depth, min_samples):
    """Build a decision tree recursively.
    Returns a dictionary: either a LEAF {"label": ...} or a SPLIT NODE {"feature": ..., "children": ...}"""

    # The most common label in this node (this is what a leaf predicts)
    majority = int(data["target"].mode()[0])

    # ---- STOPPING CONDITIONS: make a leaf ----
    if (data["target"].nunique() == 1        # 1) the node is pure
            or len(features_left) == 0       # 2) no features left to split on
            or depth >= max_depth            # 3) the tree is deep enough
            or len(data) < min_samples):     # 4) too few rows in this node
        return {"label": majority, "n": len(data)}

    # ---- CHOOSE THE BEST FEATURE: the one with the biggest gain ----
    gains = {f: split_gain(data, f, impurity_fn) for f in features_left}
    best_feature = max(gains, key=gains.get)

    # 5) if even the best split does not help, make a leaf
    if gains[best_feature] <= 0:
        return {"label": majority, "n": len(data)}

    # ---- SPLIT: one branch per category value, then build each branch ----
    # This feature cannot be used again below this node
    remaining = [f for f in features_left if f != best_feature]
    children = {}
    for value, group in data.groupby(best_feature):
        children[value] = build_tree(group, remaining, impurity_fn,
                                     depth + 1, max_depth, min_samples)

    # "default" is used later if a test row has a value this node never saw
    return {"feature": best_feature, "default": majority,
            "n": len(data), "children": children}


def count_leaves(tree):
    """Count how many leaves (final predictions) a tree has."""
    if "feature" not in tree:
        return 1
    return sum(count_leaves(child) for child in tree["children"].values())


# Settings for both trees (same settings, so the comparison is fair)
MAX_DEPTH = 4
MIN_SAMPLES = 50

# Build one tree with Gini and one with Entropy / Information Gain
# This can take about 1 to 3 minutes
tree_gini = build_tree(train_df, features, gini, 0, MAX_DEPTH, MIN_SAMPLES)
tree_entropy = build_tree(train_df, features, entropy, 0, MAX_DEPTH, MIN_SAMPLES)

# Show a first look at both trees
print("Gini tree    -> root feature:", tree_gini["feature"], "| leaves:", count_leaves(tree_gini))
print("Entropy tree -> root feature:", tree_entropy["feature"], "| leaves:", count_leaves(tree_entropy))

Gini tree    -> root feature: ord_2 | leaves: 511
Entropy tree -> root feature: ord_2 | leaves: 518


In [15]:
def predict_row(tree, row):
    """Predict the label of ONE row by walking down the tree."""
    # Keep going while we are at a split node (leaves have no "feature")
    while "feature" in tree:
        value = row[tree["feature"]]
        if value in tree["children"]:
            tree = tree["children"][value]     # follow the matching branch
        else:
            return tree["default"]             # unseen value: use the default label
    # We reached a leaf: return its label
    return tree["label"]


def predict_all(tree, data):
    """Predict every row of a table. Returns a numpy array of predictions."""
    rows = data.to_dict("records")    # turn the table into a list of rows
    return np.array([predict_row(tree, row) for row in rows])


# The correct answers for the test set
y_test = test_df["target"].values

# Predictions of both trees on the SAME test set
pred_gini = predict_all(tree_gini, test_df)
pred_entropy = predict_all(tree_entropy, test_df)

# Accuracy = the fraction of correct predictions
acc_gini = np.mean(pred_gini == y_test)
acc_entropy = np.mean(pred_entropy == y_test)

# Baseline: the accuracy of ALWAYS guessing the most common label (0)
baseline = np.mean(y_test == 0)

# How often the two trees predict the same label
agreement = np.mean(pred_gini == pred_entropy)

print("Test accuracy - Gini tree    :", round(acc_gini, 4))
print("Test accuracy - Entropy tree :", round(acc_entropy, 4))
print("Baseline (always predict 0)  :", round(baseline, 4))
print("The two trees agree on       :", round(agreement * 100, 2), "% of the test rows")

Test accuracy - Gini tree    : 0.7007
Test accuracy - Entropy tree : 0.7008
Baseline (always predict 0)  : 0.695
The two trees agree on       : 99.94 % of the test rows


In [16]:
def show_top(tree, max_level=1, level=0, indent=""):
    """Print the top levels of a tree in a readable way."""
    # A leaf: print its prediction
    if "feature" not in tree:
        print(indent + "LEAF -> predict", tree["label"], "(n=%d)" % tree["n"])
        return
    # A split node: print the feature it splits on
    print(indent + "split on", tree["feature"], "(n=%d)" % tree["n"])
    if level >= max_level:
        return
    # Print each branch (one per category value) and go one level deeper
    for value, child in sorted(tree["children"].items()):
        print(indent + "  if", tree["feature"], "=", value, ":")
        show_top(child, max_level, level + 1, indent + "    ")


def find_differences(tree_a, tree_b, path="root"):
    """Walk down both trees together and list every node where they chose different features."""
    feature_a = tree_a.get("feature")   # None if tree_a has a leaf here
    feature_b = tree_b.get("feature")

    # If they chose different features at this node, record it and stop going deeper
    if feature_a != feature_b:
        return [(path, feature_a, feature_b, tree_a["n"])]

    # If both are leaves, there is nothing more to compare
    if feature_a is None:
        return []

    # Otherwise compare each branch below this node
    differences = []
    for value in tree_a["children"]:
        child_path = path + " > " + feature_a + "=" + str(value)
        differences += find_differences(tree_a["children"][value],
                                        tree_b["children"][value], child_path)
    return differences


print("TOP OF THE GINI TREE")
show_top(tree_gini, max_level=1)

print()
print("NODES WHERE THE TWO TREES CHOSE DIFFERENT FEATURES:")
diffs = find_differences(tree_gini, tree_entropy)
for path, feat_gini, feat_entropy, n in diffs:
    print(path, "| Gini:", feat_gini, "| Entropy:", feat_entropy, "| rows:", n)
print()
print("Number of differing nodes:", len(diffs))

TOP OF THE GINI TREE
split on ord_2 (n=240000)
  if ord_2 = Boiling Hot :
    split on ord_1 (n=48691)
  if ord_2 = Cold :
    split on ord_1 (n=27068)
  if ord_2 = Freezing :
    split on ord_1 (n=79677)
  if ord_2 = Hot :
    split on ord_1 (n=17819)
  if ord_2 = Lava Hot :
    split on ord_1 (n=51090)
  if ord_2 = Warm :
    split on ord_1 (n=15655)

NODES WHERE THE TWO TREES CHOSE DIFFERENT FEATURES:
root > ord_2=Cold > ord_1=Master > nom_0=Red | Gini: nom_3 | Entropy: nom_4 | rows: 631
root > ord_2=Freezing > ord_1=Contributor > nom_0=Green | Gini: nom_4 | Entropy: nom_1 | rows: 4910
root > ord_2=Warm > ord_1=Expert > nom_3=China | Gini: nom_0 | Entropy: nom_1 | rows: 176
root > ord_2=Warm > ord_1=Expert > nom_3=India | Gini: nom_4 | Entropy: nom_1 | rows: 111
root > ord_2=Warm > ord_1=Novice > nom_0=Blue | Gini: nom_4 | Entropy: nom_1 | rows: 2093

Number of differing nodes: 5


# Explanation: Comparing Gini Impurity and Entropy / Information Gain

I built a Decision Tree classifier from scratch with Pandas and NumPy (no sklearn tree). The data (300,000 rows, 9 categorical features, binary target) was split 80% for training (240,000 rows) and 20% for testing (60,000 rows). At each node the code calculates, for every unused feature, the **gain** = impurity before the split − weighted average impurity of the branches (one branch for each category value). The feature with the largest gain is chosen, and the process repeats recursively. A node becomes a leaf (predicting its majority label) when it is pure, has no features left, reaches the maximum depth of 4, has fewer than 50 rows, or the best gain is not positive. Both trees used exactly the same settings, so the only difference is the splitting criterion.

- **Gini impurity** = 1 − Σ p². The gain is the reduction in Gini impurity.
- **Entropy** = −Σ p·log₂(p). The gain is the **Information Gain**.

## Results
| | Gini tree | Entropy tree |
|---|---|---|
| Root feature | ord_2 | ord_2 |
| Number of leaves | 511 | 518 |
| Test accuracy | 0.7007 | 0.7008 |

The baseline of always predicting the most common class (0) gives 0.695. The two trees made the same prediction on 99.94% of the test rows.

## Comparison
- **Feature selection:** At the root, both criteria ranked the features in the same order (ord_2 > ord_1 > nom_0 > nom_4 > nom_1 > nom_3 > nom_2 > bin_4 > bin_3), and both chose ord_2. The Information Gain values were larger (for example 0.01709 versus 0.01002 for ord_2), because entropy is measured on a larger scale than Gini, but the ranking was the same. The second level of both trees also used ord_1 under every ord_2 value.
- **Tree structure:** The trees differ in only 5 nodes deep inside the tree (for example, under ord_2=Cold > ord_1=Master > nom_0=Red, Gini chose nom_3 while Entropy chose nom_4). These nodes are small (111 to 4,910 rows out of 240,000). The Entropy tree has 7 more leaves (518 versus 511) because at these 5 nodes it chose features with more categories (for example nom_1 has 6 values, while Gini chose nom_0 with 3 values or nom_4 with 4 values), and each category value becomes one leaf.
- **Predictions:** Because the structures are almost the same, the accuracies are almost the same (0.7007 versus 0.7008), and the predictions agree on 99.94% of the test rows.
- **Why they are similar:** Gini and Entropy are both measures of how mixed a group is. They are both 0 for a pure group and largest for a 50/50 group, and they have a very similar shape, so they usually prefer the same splits. They differ only when two features have very close gains, which is what happened in the 5 nodes. Entropy penalizes mixed groups slightly more, and it needs a logarithm, so it is slightly more expensive to compute, while Gini is simpler.

## Conclusion
For this dataset, the choice between Gini and Entropy made almost no difference. Both trees are only about 0.5 percentage points more accurate than always guessing the majority class. The reason is that the features carry very little information about the target (the best gain at the root is only about 0.01 for Gini), so the limit comes from the data rather than from the splitting criterion.